In [0]:
# Define the catalog and schemas used by our pipeline.
catalog = "gpo_analytics"
bronze_schema = "bronze"
silver_schema = "silver"

# Confirm that the Silver schema exists.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

# Display the Bronze tables available for transformation.
display(spark.sql(f"SHOW TABLES IN {catalog}.{bronze_schema}"))

database,tableName,isTemporary
bronze,adverseevents,false
bronze,drugdispensation,false
bronze,investigator,false
bronze,labresults,false
bronze,randomization,false
bronze,shipment,false
bronze,site,false
bronze,study,false
bronze,subject,false
bronze,visit,false


In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze study table.
study_bronze = spark.table("gpo_analytics.bronze.study")

# 2. Clean text fields and convert columns to appropriate data types.
study_silver = (
    study_bronze
    .select(
        F.trim(F.col("StudyID")).alias("StudyID"),
        F.trim(F.col("ProtocolID")).alias("ProtocolID"),
        F.trim(F.col("StudyName")).alias("StudyName"),
        F.trim(F.col("Phase")).alias("Phase"),
        F.trim(F.col("TherapeuticArea")).alias("TherapeuticArea"),
        F.trim(F.col("Status")).alias("Status"),
        F.trim(F.col("PlannedEnrollment")).cast("int").alias("PlannedEnrollment"),
        F.to_date(F.col("StudyStartDate"), "yyyy-MM-dd").alias("StudyStartDate"),
        F.to_date(F.col("StudyEndDate"), "yyyy-MM-dd").alias("StudyEndDate")
    )
)

# 3. Save the transformed data as a Silver Delta table.
(
    study_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gpo_analytics.silver.study")
)

# 4. Display 10 rows from the saved Silver table.
display(spark.table("gpo_analytics.silver.study").limit(10))

StudyID,ProtocolID,StudyName,Phase,TherapeuticArea,Status,PlannedEnrollment,StudyStartDate,StudyEndDate
ST001,GPO-ONC-101,Oncology Clinical Study 01,Phase I,Oncology,Recruiting,162,2025-04-28,2027-12-19
ST002,GPO-IMM-102,Immunology Clinical Study 02,Phase II,Immunology,Recruiting,331,2024-04-19,2028-07-30
ST003,GPO-RAR-103,Rare Disease Clinical Study 03,Phase I,Rare Disease,Active,161,2025-02-03,2026-08-09
ST004,GPO-ONC-104,Oncology Clinical Study 04,Phase I,Oncology,Recruiting,165,2024-07-31,2028-12-05
ST005,GPO-RAR-105,Rare Disease Clinical Study 05,Phase I,Rare Disease,Recruiting,473,2025-04-14,2028-02-26
ST006,GPO-RAR-106,Rare Disease Clinical Study 06,Phase II,Rare Disease,Active,498,2024-07-16,2026-05-21
ST007,GPO-CAR-107,Cardiology Clinical Study 07,Phase III,Cardiology,Recruiting,523,2024-05-08,2027-07-03
ST008,GPO-IMM-108,Immunology Clinical Study 08,Phase III,Immunology,Active,298,2023-07-20,2028-10-12
ST009,GPO-IMM-109,Immunology Clinical Study 09,Phase II,Immunology,Completed,495,2024-12-06,2027-03-18
ST010,GPO-ONC-110,Oncology Clinical Study 10,Phase I,Oncology,Recruiting,515,2024-08-20,2027-05-01


In [0]:
from pyspark.sql import functions as F

# Read the Bronze and Silver study tables.
bronze = spark.table("gpo_analytics.bronze.study")
silver = spark.table("gpo_analytics.silver.study")

# Count records in each table.
bronze_count = bronze.count()
silver_count = silver.count()

# Count missing StudyIDs.
missing_ids = silver.filter(
    F.col("StudyID").isNull() | (F.trim(F.col("StudyID")) == "")
).count()

# Count duplicate StudyIDs.
duplicate_ids = (
    silver.groupBy("StudyID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Display the validation results.
print(f"Bronze row count: {bronze_count}")
print(f"Silver row count: {silver_count}")
print(f"Missing StudyIDs: {missing_ids}")
print(f"Duplicate StudyIDs: {duplicate_ids}")

# Display the Silver column names and data types.
silver.printSchema()

Bronze row count: 20
Silver row count: 20
Missing StudyIDs: 0
Duplicate StudyIDs: 0
root
 |-- StudyID: string (nullable = true)
 |-- ProtocolID: string (nullable = true)
 |-- StudyName: string (nullable = true)
 |-- Phase: string (nullable = true)
 |-- TherapeuticArea: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- PlannedEnrollment: integer (nullable = true)
 |-- StudyStartDate: date (nullable = true)
 |-- StudyEndDate: date (nullable = true)



In [0]:
# Read the Bronze site table.
site_bronze = spark.table("gpo_analytics.bronze.site")

# Print the exact column names.
print(site_bronze.columns)

['SiteID', 'SiteName', 'Country', 'City', 'StudyID', 'SiteStatus', 'ActivationDate', '_source_file', '_ingested_at']


In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze site table.
site_bronze = spark.table("gpo_analytics.bronze.site")

# 2. Clean text columns and convert the activation date.
site_silver = (
    site_bronze
    .select(
        F.trim(F.col("SiteID")).alias("SiteID"),
        F.trim(F.col("SiteName")).alias("SiteName"),
        F.trim(F.col("Country")).alias("Country"),
        F.trim(F.col("City")).alias("City"),
        F.trim(F.col("StudyID")).alias("StudyID"),
        F.trim(F.col("SiteStatus")).alias("SiteStatus"),
        F.to_date(F.col("ActivationDate"), "yyyy-MM-dd").alias("ActivationDate")
    )
)

# 3. Save the transformed data as a Silver Delta table.
(
    site_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gpo_analytics.silver.site")
)

# 4. Display 10 rows for inspection.
display(spark.table("gpo_analytics.silver.site").limit(10))

SiteID,SiteName,Country,City,StudyID,SiteStatus,ActivationDate
SITE001,Berlin Clinical Research Site 001,Germany,Berlin,ST015,Active,2025-06-09
SITE002,Madrid Clinical Research Site 002,Spain,Madrid,ST019,Active,2025-08-24
SITE003,Sydney Clinical Research Site 003,Australia,Sydney,ST008,Active,2026-05-22
SITE004,Bengaluru Clinical Research Site 004,India,Bengaluru,ST009,Closed,2024-03-31
SITE005,New York Clinical Research Site 005,USA,New York,ST019,Active,2024-05-25
SITE006,New York Clinical Research Site 006,USA,New York,ST002,Initiated,2024-09-22
SITE007,Madrid Clinical Research Site 007,Spain,Madrid,ST016,Active,2023-10-17
SITE008,Berlin Clinical Research Site 008,Germany,Berlin,ST010,Closed,2023-08-06
SITE009,Tokyo Clinical Research Site 009,Japan,Tokyo,ST014,Closed,2024-09-30
SITE010,Bengaluru Clinical Research Site 010,India,Bengaluru,ST007,Active,2023-12-02


In [0]:
from pyspark.sql import functions as F

# 1. Read the original Bronze subject table.
subject_bronze = spark.table("gpo_analytics.bronze.subject")

# 2. Clean the columns and assign appropriate data types.
subject_silver = subject_bronze.select(
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("StudyID")).alias("StudyID"),
    F.trim(F.col("SiteID")).alias("SiteID"),
    F.trim(F.col("Gender")).alias("Gender"),
    F.col("Age").cast("int").alias("Age"),
    F.trim(F.col("Disease")).alias("Disease"),
    F.to_date(F.col("ScreenDate"), "yyyy-MM-dd").alias("ScreenDate"),
    F.trim(F.col("SubjectStatus")).alias("SubjectStatus"),
    F.to_date(F.col("EnrollmentDate"), "yyyy-MM-dd").alias("EnrollmentDate")
)

# 3. Save the transformed data as a Silver Delta table.
subject_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.subject")

# 4. Display a sample of the saved table.
display(spark.table("gpo_analytics.silver.subject").limit(10))

SubjectID,StudyID,SiteID,Gender,Age,Disease,ScreenDate,SubjectStatus,EnrollmentDate
SUB00001,ST017,SITE018,Male,57,Inflammatory Disease,2026-01-22,Active,2026-01-26
SUB00002,ST001,SITE030,Female,80,Solid Tumor,2026-09-18,Active,2026-10-01
SUB00003,ST001,SITE045,Unknown,41,Solid Tumor,2025-06-15,Active,2025-06-28
SUB00004,ST001,SITE030,Other,19,Solid Tumor,2023-12-02,Withdrawn,2023-12-23
SUB00005,ST020,SITE042,Male,50,Solid Tumor,2024-12-20,Active,2025-01-06
SUB00006,ST016,SITE007,Other,23,Inflammatory Disease,2024-05-01,Enrolled,2024-05-12
SUB00007,ST013,SITE068,Unknown,51,Neurological Disorder,2024-11-19,Screen Failed,null
SUB00008,ST013,SITE094,Other,28,Neurological Disorder,2025-04-27,Active,2025-05-09
SUB00009,ST019,SITE048,Female,72,Cardiovascular Disease,2024-04-21,Screened,null
SUB00010,ST016,SITE096,Female,40,Inflammatory Disease,2025-03-25,Screened,null


In [0]:
from pyspark.sql import functions as F

# Read both tables.
bronze = spark.table("gpo_analytics.bronze.subject")
silver = spark.table("gpo_analytics.silver.subject")

# Compare row counts.
print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

# Count missing and duplicate SubjectIDs.
print(
    "Missing SubjectIDs:",
    silver.filter(
        F.col("SubjectID").isNull() |
        (F.trim(F.col("SubjectID")) == "")
    ).count()
)

print(
    "Duplicate SubjectIDs:",
    silver.groupBy("SubjectID")
          .count()
          .filter(F.col("count") > 1)
          .count()
)

# Confirm the resulting data types.
silver.printSchema()

Bronze row count: 3000
Silver row count: 3000
Missing SubjectIDs: 0
Duplicate SubjectIDs: 0
root
 |-- SubjectID: string (nullable = true)
 |-- StudyID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Disease: string (nullable = true)
 |-- ScreenDate: date (nullable = true)
 |-- SubjectStatus: string (nullable = true)
 |-- EnrollmentDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Read the original Bronze visit table.
visit_bronze = spark.table("gpo_analytics.bronze.visit")

# 2. Clean text columns and convert VisitDate to a date.
visit_silver = visit_bronze.select(
    F.trim(F.col("VisitID")).alias("VisitID"),
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("VisitName")).alias("VisitName"),
    F.to_date(F.col("VisitDate"), "yyyy-MM-dd").alias("VisitDate"),
    F.trim(F.col("Status")).alias("Status")
)

# 3. Save the transformed table in Silver.
visit_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.visit")

# 4. Display ten sample rows.
display(spark.table("gpo_analytics.silver.visit").limit(10))

VisitID,SubjectID,VisitName,VisitDate,Status
V0000001,SUB00001,Screening,2026-01-22,Completed
V0000002,SUB00001,Baseline,2026-01-29,Cancelled
V0000003,SUB00001,Week 4,2026-02-19,Completed
V0000004,SUB00001,Week 8,2026-03-19,Completed
V0000005,SUB00001,Week 12,2026-04-16,Completed
V0000006,SUB00002,Screening,2026-09-18,Completed
V0000007,SUB00002,Baseline,2026-09-25,Completed
V0000008,SUB00003,Screening,2025-06-15,Completed
V0000009,SUB00003,Baseline,2025-06-22,Scheduled
V0000010,SUB00003,Week 4,2025-07-13,Missed


In [0]:
from pyspark.sql import functions as F

# 1. Load the Bronze and Silver Visit tables.
visit_bronze = spark.table("gpo_analytics.bronze.visit")
visit_silver = spark.table("gpo_analytics.silver.visit")

# 2. Compare the total number of rows.
bronze_count = visit_bronze.count()
silver_count = visit_silver.count()

# 3. Count missing or blank VisitIDs in Silver.
missing_visit_ids = visit_silver.filter(
    F.col("VisitID").isNull() |
    (F.trim(F.col("VisitID")) == "")
).count()

# 4. Count extra rows caused by duplicate VisitIDs.
duplicate_visit_ids = (
    visit_silver
    .groupBy("VisitID")
    .count()
    .filter(F.col("count") > 1)
    .agg(F.sum(F.col("count") - 1).alias("duplicates"))
    .first()["duplicates"]
    or 0
)

# 5. Count visits with missing or blank SubjectIDs.
missing_subject_ids = visit_silver.filter(
    F.col("SubjectID").isNull() |
    (F.trim(F.col("SubjectID")) == "")
).count()

# 6. Find visits whose SubjectID does not exist in Silver Subject.
valid_subjects = (
    spark.table("gpo_analytics.silver.subject")
    .select("SubjectID")
    .filter(F.col("SubjectID").isNotNull())
    .distinct()
)

unmatched_subjects = (
    visit_silver
    .join(valid_subjects, on="SubjectID", how="left_anti")
    .filter(
        F.col("SubjectID").isNotNull() &
        (F.trim(F.col("SubjectID")) != "")
    )
    .count()
)

# 7. Print the validation results.
print(f"Bronze Visit row count: {bronze_count}")
print(f"Silver Visit row count: {silver_count}")
print(f"Missing VisitIDs: {missing_visit_ids}")
print(f"Duplicate VisitID extra rows: {duplicate_visit_ids}")
print(f"Missing SubjectIDs: {missing_subject_ids}")
print(f"Unmatched SubjectIDs: {unmatched_subjects}")

Bronze Visit row count: 14398
Silver Visit row count: 14398
Missing VisitIDs: 0
Duplicate VisitID extra rows: 0
Missing SubjectIDs: 0
Unmatched SubjectIDs: 0


In [0]:
from pyspark.sql import functions as F

# 1. Read the original Bronze table.
ae_bronze = spark.table("gpo_analytics.bronze.adverseevents")

# 2. Select business columns and trim text values.
ae_silver = ae_bronze.select(
    F.trim(F.col("AEID")).alias("AEID"),
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("Event")).alias("Event"),
    F.trim(F.col("Severity")).alias("Severity"),
    F.trim(F.col("Serious")).alias("Serious"),
    F.trim(F.col("Relationship")).alias("Relationship"),
    F.trim(F.col("Outcome")).alias("Outcome")
)

# 3. Save the transformed data as a Silver Delta table.
ae_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.adverseevents")

# 4. Display a sample of the created table.
display(spark.table("gpo_analytics.silver.adverseevents").limit(10))

AEID,SubjectID,Event,Severity,Serious,Relationship,Outcome
AE000001,SUB00500,Diarrhea,Grade 1,No,Related,Ongoing
AE000002,SUB01155,Elevated ALT,Grade 2,Yes,Related,Fatal
AE000003,SUB00424,Diarrhea,Grade 3,Yes,Related,Recovered with Sequelae
AE000004,SUB01904,Headache,Grade 1,No,Related,Recovered with Sequelae
AE000005,SUB01758,Headache,Grade 3,Yes,Not Assessed,Recovering
AE000006,SUB00247,Fatigue,Grade 4,Yes,Related,Fatal
AE000007,SUB01297,Nausea,Grade 3,No,Possibly Related,Recovered with Sequelae
AE000008,SUB00859,Neutropenia,Grade 2,No,Not Assessed,Recovered
AE000009,SUB01690,Injection-site reaction,Grade 2,No,Not Assessed,Ongoing
AE000010,SUB02793,Elevated ALT,Grade 2,No,Unrelated,Recovering


In [0]:
from pyspark.sql import functions as F

# 1. Load Bronze and Silver adverse event tables.
ae_bronze = spark.table("gpo_analytics.bronze.adverseevents")
ae_silver = spark.table("gpo_analytics.silver.adverseevents")

# 2. Compare row counts.
print("Bronze Adverse Events:", ae_bronze.count())
print("Silver Adverse Events:", ae_silver.count())

# 3. Count missing or blank AEIDs.
missing_aeids = ae_silver.filter(
    F.col("AEID").isNull() |
    (F.trim(F.col("AEID")) == "")
).count()

# 4. Count duplicate AEID extra rows.
duplicate_aeids = (
    ae_silver
    .groupBy("AEID")
    .count()
    .filter(F.col("count") > 1)
    .agg(F.sum(F.col("count") - 1).alias("duplicates"))
    .first()["duplicates"]
    or 0
)

# 5. Count missing or blank SubjectIDs.
missing_subject_ids = ae_silver.filter(
    F.col("SubjectID").isNull() |
    (F.trim(F.col("SubjectID")) == "")
).count()

# 6. Build the list of valid subject IDs from Silver.
valid_subjects = (
    spark.table("gpo_analytics.silver.subject")
    .select("SubjectID")
    .filter(
        F.col("SubjectID").isNotNull() &
        (F.trim(F.col("SubjectID")) != "")
    )
    .distinct()
)

# 7. Count nonblank SubjectIDs that do not match a Silver subject.
unmatched_subjects = (
    ae_silver
    .join(valid_subjects, on="SubjectID", how="left_anti")
    .filter(
        F.col("SubjectID").isNotNull() &
        (F.trim(F.col("SubjectID")) != "")
    )
    .count()
)

# 8. Display the validation results.
print("Missing AEIDs:", missing_aeids)
print("Duplicate AEID extra rows:", duplicate_aeids)
print("Missing SubjectIDs:", missing_subject_ids)
print("Unmatched SubjectIDs:", unmatched_subjects)

Bronze Adverse Events: 1200
Silver Adverse Events: 1200
Missing AEIDs: 0
Duplicate AEID extra rows: 0
Missing SubjectIDs: 0
Unmatched SubjectIDs: 0


In [0]:
from pyspark.sql import functions as F

# 1. Read the original Bronze table.
drug_bronze = spark.table("gpo_analytics.bronze.drugdispensation")

# 2. Clean text fields and convert DispenseDate to a date.
drug_silver = drug_bronze.select(
    F.trim(F.col("DispenseID")).alias("DispenseID"),
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("Drug")).alias("Drug"),
    F.trim(F.col("LotNumber")).alias("LotNumber"),
    F.trim(F.col("Dose")).alias("Dose"),
    F.to_date(F.col("DispenseDate"), "yyyy-MM-dd").alias("DispenseDate")
)

# 3. Save the transformed data as a Silver Delta table.
drug_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.drugdispensation")

# 4. Display a sample of the saved Silver table.
display(spark.table("gpo_analytics.silver.drugdispensation").limit(10))

DispenseID,SubjectID,Drug,LotNumber,Dose,DispenseDate
D000001,SUB02516,Placebo,LOT1120,25 mg,2025-11-09
D000002,SUB02491,GPO-201,LOT1043,100 mg,2026-08-13
D000003,SUB01704,GPO-305,LOT1097,1 Tablet,2026-02-03
D000004,SUB02771,GPO-201,LOT1068,50 mg,2026-06-18
D000005,SUB02799,GPO-305,LOT1092,1 Tablet,2025-05-08
D000006,SUB00274,Placebo,LOT1050,100 mg,2026-02-24
D000007,SUB02577,GPO-305,LOT1093,1 Tablet,2026-04-12
D000008,SUB01610,Placebo,LOT1116,50 mg,2025-10-04
D000009,SUB02377,GPO-201,LOT1067,25 mg,2026-09-28
D000010,SUB01198,GPO-201,LOT1077,100 mg,2026-10-01


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("gpo_analytics.bronze.drugdispensation")
silver = spark.table("gpo_analytics.silver.drugdispensation")
subjects = spark.table("gpo_analytics.silver.subject")

print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

print("Missing DispenseID:", silver.filter(
    F.col("DispenseID").isNull() | (F.trim(F.col("DispenseID")) == "")
).count())

print("Duplicate DispenseID extra rows:", 
    silver.count() - silver.select("DispenseID").distinct().count())

print("Missing SubjectID:", silver.filter(
    F.col("SubjectID").isNull() | (F.trim(F.col("SubjectID")) == "")
).count())

unmatched_subjects = silver.join(
    subjects.select("SubjectID"),
    on="SubjectID",
    how="left_anti"
)

print("Unmatched SubjectID:", unmatched_subjects.count())

silver.printSchema()

Bronze row count: 4500
Silver row count: 4500
Missing DispenseID: 0
Duplicate DispenseID extra rows: 0
Missing SubjectID: 0
Unmatched SubjectID: 0
root
 |-- DispenseID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- Drug: string (nullable = true)
 |-- LotNumber: string (nullable = true)
 |-- Dose: string (nullable = true)
 |-- DispenseDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

randomization_bronze = spark.table(
    "gpo_analytics.bronze.randomization"
)

randomization_silver = randomization_bronze.select(
    F.trim(F.col("RandomizationID")).alias("RandomizationID"),
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("TreatmentArm")).alias("TreatmentArm"),
    F.to_date(
        F.col("RandomizationDate"), "yyyy-MM-dd"
    ).alias("RandomizationDate")
)

randomization_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.randomization")

display(
    spark.table("gpo_analytics.silver.randomization").limit(10)
)

RandomizationID,SubjectID,TreatmentArm,RandomizationDate
R000001,SUB01102,Placebo,2025-08-15
R000002,SUB01802,Treatment B,2024-05-12
R000003,SUB01438,Treatment A,2023-10-14
R000004,SUB02667,Placebo,2024-08-09
R000005,SUB01757,Treatment B,2025-01-20
R000006,SUB01378,Placebo,2025-03-11
R000007,SUB00573,Placebo,2026-01-05
R000008,SUB02603,Treatment A,2025-07-07
R000009,SUB00707,Treatment A,2025-02-20
R000010,SUB00346,Placebo,2026-03-08


In [0]:
from pyspark.sql import functions as F

bronze = spark.table("gpo_analytics.bronze.randomization")
silver = spark.table("gpo_analytics.silver.randomization")
subjects = spark.table("gpo_analytics.silver.subject")

print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

print(
    "Missing RandomizationID:",
    silver.filter(
        F.col("RandomizationID").isNull()
        | (F.trim(F.col("RandomizationID")) == "")
    ).count()
)

print(
    "Duplicate RandomizationID extra rows:",
    silver.count() - silver.select("RandomizationID").distinct().count()
)

print(
    "Missing SubjectID:",
    silver.filter(
        F.col("SubjectID").isNull()
        | (F.trim(F.col("SubjectID")) == "")
    ).count()
)

unmatched_subjects = silver.join(
    subjects.select("SubjectID"),
    on="SubjectID",
    how="left_anti"
)

print("Unmatched SubjectID:", unmatched_subjects.count())

silver.printSchema()

Bronze row count: 1915
Silver row count: 1915
Missing RandomizationID: 0
Duplicate RandomizationID extra rows: 0
Missing SubjectID: 0
Unmatched SubjectID: 0
root
 |-- RandomizationID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- TreatmentArm: string (nullable = true)
 |-- RandomizationDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze shipment table
shipment_bronze = spark.table(
    "gpo_analytics.bronze.shipment"
)

# 2. Clean text columns and convert date columns
shipment_silver = shipment_bronze.select(
    F.trim(F.col("ShipmentID")).alias("ShipmentID"),
    F.trim(F.col("LotNumber")).alias("LotNumber"),
    F.trim(F.col("Depot")).alias("Depot"),
    F.trim(F.col("SiteID")).alias("SiteID"),
    F.to_date(
        F.col("ShipmentDate"), "yyyy-MM-dd"
    ).alias("ShipmentDate"),
    F.trim(F.col("Status")).alias("Status"),
    F.to_date(
        F.col("ExpectedDeliveryDate"), "yyyy-MM-dd"
    ).alias("ExpectedDeliveryDate"),
    F.to_date(
        F.col("ActualDeliveryDate"), "yyyy-MM-dd"
    ).alias("ActualDeliveryDate")
)

# 3. Save the transformed data as a Silver Delta table
shipment_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.shipment")

# 4. Inspect the newly created table
display(
    spark.table("gpo_analytics.silver.shipment").limit(10)
)

spark.table("gpo_analytics.silver.shipment").printSchema()

ShipmentID,LotNumber,Depot,SiteID,ShipmentDate,Status,ExpectedDeliveryDate,ActualDeliveryDate
SHIP00001,LOT1084,Depot APAC,SITE053,2025-01-26,Delivered,2025-02-01,2025-02-04
SHIP00002,LOT1114,Depot Europe,SITE077,2025-01-08,Delivered,2025-01-16,2025-01-21
SHIP00003,LOT1061,Depot North,SITE064,2025-12-03,Delivered,2025-12-10,2025-12-21
SHIP00004,LOT1076,Depot South,SITE072,2024-02-25,Delivered,2024-03-09,2024-03-09
SHIP00005,LOT1030,Depot South,SITE093,2025-01-27,Delivered,2025-02-04,2025-01-31
SHIP00006,LOT1004,Depot APAC,SITE007,2025-07-28,Delivered,2025-07-31,2025-08-08
SHIP00007,LOT1061,Depot South,SITE023,2024-06-29,Pending,2024-07-05,null
SHIP00008,LOT1024,Depot North,SITE062,2025-03-03,Delivered,2025-03-14,2025-03-23
SHIP00009,LOT1067,Depot North,SITE073,2024-07-01,In Transit,2024-07-05,null
SHIP00010,LOT1098,Depot North,SITE003,2024-06-12,Pending,2024-06-26,null


root
 |-- ShipmentID: string (nullable = true)
 |-- LotNumber: string (nullable = true)
 |-- Depot: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- ShipmentDate: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- ExpectedDeliveryDate: date (nullable = true)
 |-- ActualDeliveryDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Load the Bronze, Silver, and site tables
bronze = spark.table("gpo_analytics.bronze.shipment")
silver = spark.table("gpo_analytics.silver.shipment")
sites = spark.table("gpo_analytics.silver.site")

# 2. Compare row counts
print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

# 3. Check for missing ShipmentID values
print(
    "Missing ShipmentID:",
    silver.filter(
        F.col("ShipmentID").isNull()
        | (F.trim(F.col("ShipmentID")) == "")
    ).count()
)

# 4. Check for duplicate ShipmentID values
print(
    "Duplicate ShipmentID extra rows:",
    silver.count() - silver.select("ShipmentID").distinct().count()
)

# 5. Check for missing SiteID values
print(
    "Missing SiteID:",
    silver.filter(
        F.col("SiteID").isNull()
        | (F.trim(F.col("SiteID")) == "")
    ).count()
)

# 6. Find shipments whose SiteID does not exist in Silver site
unmatched_sites = silver.join(
    sites.select("SiteID"),
    on="SiteID",
    how="left_anti"
)

print("Unmatched SiteID:", unmatched_sites.count())

# 7. Check whether non-null source dates failed conversion
for column in [
    "ShipmentDate",
    "ExpectedDeliveryDate",
    "ActualDeliveryDate"
]:
    invalid_count = bronze.filter(
        F.col(column).isNotNull()
        & (F.trim(F.col(column)) != "")
        & F.to_date(F.col(column), "yyyy-MM-dd").isNull()
    ).count()

    print(f"Invalid {column} values:", invalid_count)

# 8. Check delivery-status consistency
delivered_missing_date = silver.filter(
    (F.lower(F.trim(F.col("Status"))) == "delivered")
    & F.col("ActualDeliveryDate").isNull()
).count()

print("Delivered shipments missing actual date:", delivered_missing_date)

not_delivered_with_date = silver.filter(
    (F.lower(F.trim(F.col("Status"))) != "delivered")
    & F.col("Status").isNotNull()
    & F.col("ActualDeliveryDate").isNotNull()
).count()

print("Non-delivered shipments with actual date:", not_delivered_with_date)

# 9. Confirm Silver data types
silver.printSchema()

Bronze row count: 600
Silver row count: 600
Missing ShipmentID: 0
Duplicate ShipmentID extra rows: 0
Missing SiteID: 0
Unmatched SiteID: 0
Invalid ShipmentDate values: 0
Invalid ExpectedDeliveryDate values: 0
Invalid ActualDeliveryDate values: 0
Delivered shipments missing actual date: 0
Non-delivered shipments with actual date: 0
root
 |-- ShipmentID: string (nullable = true)
 |-- LotNumber: string (nullable = true)
 |-- Depot: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- ShipmentDate: date (nullable = true)
 |-- Status: string (nullable = true)
 |-- ExpectedDeliveryDate: date (nullable = true)
 |-- ActualDeliveryDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze investigator table
investigator_bronze = spark.table(
    "gpo_analytics.bronze.investigator"
)

# 2. Clean the business columns
investigator_silver = investigator_bronze.select(
    F.trim(F.col("InvestigatorID")).alias("InvestigatorID"),
    F.trim(F.col("SiteID")).alias("SiteID"),
    F.trim(F.col("InvestigatorName")).alias("InvestigatorName"),
    F.trim(F.col("Specialty")).alias("Specialty")
)

# 3. Save as a Silver Delta table
investigator_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.investigator")

# 4. Preview the newly created table
display(
    spark.table("gpo_analytics.silver.investigator").limit(10)
)

# 5. Confirm the column names and data types
spark.table(
    "gpo_analytics.silver.investigator"
).printSchema()

InvestigatorID,SiteID,InvestigatorName,Specialty
INV001,SITE001,Dr Synthetic Investigator 001,Immunology
INV002,SITE002,Dr Synthetic Investigator 002,Cardiology
INV003,SITE003,Dr Synthetic Investigator 003,Immunology
INV004,SITE004,Dr Synthetic Investigator 004,Immunology
INV005,SITE005,Dr Synthetic Investigator 005,Cardiology
INV006,SITE006,Dr Synthetic Investigator 006,Immunology
INV007,SITE007,Dr Synthetic Investigator 007,Immunology
INV008,SITE008,Dr Synthetic Investigator 008,Oncology
INV009,SITE009,Dr Synthetic Investigator 009,Rare Disease
INV010,SITE010,Dr Synthetic Investigator 010,Cardiology


root
 |-- InvestigatorID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- InvestigatorName: string (nullable = true)
 |-- Specialty: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Load the Bronze, Silver, and site tables
bronze = spark.table("gpo_analytics.bronze.investigator")
silver = spark.table("gpo_analytics.silver.investigator")
sites = spark.table("gpo_analytics.silver.site")

# 2. Compare row counts
print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

# 3. Check for missing InvestigatorID values
print(
    "Missing InvestigatorID:",
    silver.filter(
        F.col("InvestigatorID").isNull()
        | (F.trim(F.col("InvestigatorID")) == "")
    ).count()
)

# 4. Check for duplicate InvestigatorID values
print(
    "Duplicate InvestigatorID extra rows:",
    silver.count() - silver.select("InvestigatorID").distinct().count()
)

# 5. Check for missing SiteID values
print(
    "Missing SiteID:",
    silver.filter(
        F.col("SiteID").isNull()
        | (F.trim(F.col("SiteID")) == "")
    ).count()
)

# 6. Check whether each investigator references an existing Silver site
unmatched_sites = silver.join(
    sites.select("SiteID"),
    on="SiteID",
    how="left_anti"
)

print("Unmatched SiteID:", unmatched_sites.count())

# 7. Display the Silver schema
silver.printSchema()

Bronze row count: 100
Silver row count: 100
Missing InvestigatorID: 0
Duplicate InvestigatorID extra rows: 0
Missing SiteID: 0
Unmatched SiteID: 0
root
 |-- InvestigatorID: string (nullable = true)
 |-- SiteID: string (nullable = true)
 |-- InvestigatorName: string (nullable = true)
 |-- Specialty: string (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Read the Bronze labresults table
lab_bronze = spark.table(
    "gpo_analytics.bronze.labresults"
)

# 2. Clean text fields and convert LabDate to a date
lab_silver = lab_bronze.select(
    F.trim(F.col("LabID")).alias("LabID"),
    F.trim(F.col("SubjectID")).alias("SubjectID"),
    F.trim(F.col("TestName")).alias("TestName"),
    F.trim(F.col("Result")).alias("Result"),
    F.trim(F.col("Unit")).alias("Unit"),
    F.trim(F.col("Flag")).alias("Flag"),
    F.to_date(
        F.col("LabDate"), "yyyy-MM-dd"
    ).alias("LabDate")
)

# 3. Save the Silver Delta table
lab_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gpo_analytics.silver.labresults")

# 4. Preview the created table
display(
    spark.table("gpo_analytics.silver.labresults").limit(10)
)

# 5. Confirm the column data types
spark.table(
    "gpo_analytics.silver.labresults"
).printSchema()

LabID,SubjectID,TestName,Result,Unit,Flag,LabDate
L0000001,SUB00112,ALT,28.74,U/L,Normal,2026-07-05
L0000002,SUB02855,Creatinine,0.86,mg/dL,Normal,2026-03-10
L0000003,SUB01795,ALT,33.37,U/L,Normal,2026-06-06
L0000004,SUB02738,Creatinine,1.04,mg/dL,Normal,2024-10-09
L0000005,SUB00057,Hemoglobin,18.96,g/dL,High,2026-07-25
L0000006,SUB02439,Platelets,323.47,10^9/L,High,2025-01-30
L0000007,SUB02229,WBC,6.4,10^9/L,Normal,2024-02-19
L0000008,SUB00212,Hemoglobin,13.27,g/dL,Normal,2024-01-20
L0000009,SUB00863,Hemoglobin,17.15,g/dL,High,2025-06-19
L0000010,SUB02537,Platelets,248.59,10^9/L,Normal,2025-12-28


root
 |-- LabID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- TestName: string (nullable = true)
 |-- Result: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Flag: string (nullable = true)
 |-- LabDate: date (nullable = true)



In [0]:
from pyspark.sql import functions as F

# 1. Load Bronze, Silver, and subject tables
bronze = spark.table("gpo_analytics.bronze.labresults")
silver = spark.table("gpo_analytics.silver.labresults")
subjects = spark.table("gpo_analytics.silver.subject")

# 2. Compare row counts
print("Bronze row count:", bronze.count())
print("Silver row count:", silver.count())

# 3. Check for missing LabID values
print(
    "Missing LabID:",
    silver.filter(
        F.col("LabID").isNull()
        | (F.trim(F.col("LabID")) == "")
    ).count()
)

# 4. Check for duplicate LabID values
print(
    "Duplicate LabID extra rows:",
    silver.count() - silver.select("LabID").distinct().count()
)

# 5. Check for missing SubjectID values
print(
    "Missing SubjectID:",
    silver.filter(
        F.col("SubjectID").isNull()
        | (F.trim(F.col("SubjectID")) == "")
    ).count()
)

# 6. Check for SubjectID values absent from Silver subject
unmatched_subjects = silver.join(
    subjects.select("SubjectID"),
    on="SubjectID",
    how="left_anti"
)

print("Unmatched SubjectID:", unmatched_subjects.count())

# 7. Check for invalid nonblank LabDate values in Bronze
invalid_lab_dates = bronze.filter(
    F.col("LabDate").isNotNull()
    & (F.trim(F.col("LabDate")) != "")
    & F.to_date(F.col("LabDate"), "yyyy-MM-dd").isNull()
).count()

print("Invalid LabDate values:", invalid_lab_dates)

# 8. Confirm Silver schema
silver.printSchema()

Bronze row count: 10000
Silver row count: 10000
Missing LabID: 0
Duplicate LabID extra rows: 0
Missing SubjectID: 0
Unmatched SubjectID: 0
Invalid LabDate values: 0
root
 |-- LabID: string (nullable = true)
 |-- SubjectID: string (nullable = true)
 |-- TestName: string (nullable = true)
 |-- Result: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Flag: string (nullable = true)
 |-- LabDate: date (nullable = true)



In [0]:
silver_tables = [
    "study",
    "site",
    "subject",
    "visit",
    "adverseevents",
    "labresults",
    "drugdispensation",
    "randomization",
    "shipment",
    "investigator"
]

for table_name in silver_tables:
    table = spark.table(f"gpo_analytics.silver.{table_name}")
    print(f"{table_name}: {table.count()} rows")

study: 20 rows
site: 100 rows
subject: 3000 rows
visit: 14398 rows
adverseevents: 1200 rows
labresults: 10000 rows
drugdispensation: 4500 rows
randomization: 1915 rows
shipment: 600 rows
investigator: 100 rows


In [0]:
from pyspark.sql import functions as F

subjects = spark.table("gpo_analytics.silver.subject")

display(
    subjects.groupBy("SubjectStatus")
    .count()
    .orderBy("SubjectStatus")
)

SubjectStatus,count
Active,1160
Completed,484
Enrolled,520
Screen Failed,315
Screened,291
Withdrawn,230
